### MusicBrainz API

[MusicBrainz](https://musicbrainz.org) is a free, open music encyclopedia. The API needs **no account and no API key**.

Instead of building the HTTP requests by hand we use **[musicbrainzngs](https://python-musicbrainzngs.readthedocs.io/)**, the official Python wrapper. 
A wrapper is just code that handles the requests and parses the responses for us, so we can call plain Python functions and get dictionaries back. 
You can install the wrapper directly via `pip install musicbrainzngs` or add it to your personal environment if your using a package manager such as `uv`.

The only rule you must follow (see the [rate limiting docs](https://musicbrainz.org/doc/MusicBrainz_API/Rate_Limiting)):

- identify your application with a `User-Agent` by calling `set_useragent(...)` once before any request

musicbrainzngs automatically keeps you under the **~1 request per second** limit.

Documentation:
- Wrapper API: https://python-musicbrainzngs.readthedocs.io/en/latest/api/
- Search query syntax: https://musicbrainz.org/doc/MusicBrainz_API/Search

### Imports

In [10]:
# once installed, just import the package
import musicbrainzngs

In [11]:
# ADD YOUR VALUES HERE

# MusicBrainz has no API key, but it wants to know which application is calling and
# how to reach you. Arguments: application name, version, contact email or URL.
musicbrainzngs.set_useragent(
    "ditw-2025-example",        # your application name
    "0.1",                      # your application version
    "example-email@itu.dk",     # your contact email
)

#### Get information about your favourite artist

The following demonstrates a plain **search** (find the artist by name) followed by a **lookup** (use the returned id to fetch full details).

You can try with another artist if you'd like!

In [ ]:
# search MusicBrainz for your favourite artist by name
artist_search_result = musicbrainzngs.search_artists(query="The Beatles", limit=1)

# the search returns a ranked list of matches in "artist-list"; take the first one
artist = artist_search_result["artist-list"][0]

# the MBID (MusicBrainz ID) uniquely identifies this artist
artist_mbid = artist["id"]    
artist_name = artist["name"]

print(f"Unique ID: {artist_mbid}")
print(f"Artist name: {artist_name}")

Unique ID: b10bbbfc-cf9e-42e0-be17-e2c3e1d2600d
Artist name: The Beatles


In [13]:
# use the MBID to look up full information about that one artist
# "includes" allows you to ask for additional linked data. here we ask for the community rating)
artist_details = musicbrainzngs.get_artist_by_id(artist_mbid, includes=["ratings"])["artist"]

print(f"Name: {artist_details["name"]}")
print(f"Type: {artist_details["type"]}")
print(f"Country: {artist_details["country"]}")
print(f"Active: {artist_details["life-span"]["begin"]} - {artist_details["life-span"].get("end")}")
print(f"Rating: {artist_details["rating"]["rating"]} ({artist_details["rating"]["votes-count"]} votes)")

Name: The Beatles
Type: Group
Country: GB
Active: 1960-03-27 - 1970-04-10
Rating: 4.8 (86 votes)


#### Get a specific album by an artist

The following demonstrates **filtering within one artist**. A *release group* is an album independent of its many editions (CD, vinyl, remaster, ...). We browse only the release groups that belong to the artist's MBID, then keep the one album we want.

In [14]:
# browse every album-type release group that belongs to an artist
album_result = musicbrainzngs.browse_release_groups(artist=artist_mbid, release_type=["album"], limit=100)
print(f"{artist_name} has {album_result["release-group-count"]} album release groups")

# filter that list down to the single album we are interested in
album_title = "Abbey Road"
matches = [release_group for release_group in album_result["release-group-list"] if release_group["title"].lower() == album_title.lower()]

album = matches[0]
album_mbid = album["id"]
print(f"MBID: {album_mbid} \nTitle: {album["title"]} \nYear: {album["first-release-date"]}")

The Beatles has 703 album release groups
MBID: 9162580e-5df4-32de-80cc-f45a8d8a9b1d 
Title: Abbey Road 
Year: 1969-09-26


#### Get every artist in the Faroe Islands 🇫🇴

This example combines an **area filter** with **pagination**. The search query syntax has an `area:` field. Since the area name is two words, we quote it - `area:"Faroe Islands"` - otherwise it is read as `area:Faroe` plus a loose `Islands`.

- `search_artists` returns one page at a time
- `limit` is the page size (max 100)
- `offset` is how many results to skip. 

The first response also reports the total `artist-count`, so once we have it we know exactly which offsets are left to request. We collect every page, then walk through them.

In [15]:
# quote the area name so it is matched exactly as the phrase "Faroe Islands"
query = 'area:"Faroe Islands"'
limit = 100        # page size (100 is the maximum MusicBrainz allows)

# the first request also tells us the total number of matches
first_page = musicbrainzngs.search_artists(query=query, limit=limit)
total = int(first_page["artist-count"])

print(f"MusicBrainz lists {total} artists linked to the Faroe Islands")

# collect the first page plus every remaining page (offset 100, 200, ...)
pages = [first_page]
for offset in range(limit, total, limit):
    pages.append(musicbrainzngs.search_artists(query=query, limit=limit, offset=offset))

# now walk through every page and gather the artists
# we use a dictionary and drop any dublications
artists = {}
for page in pages:
    for artist in page["artist-list"]:
        artists[artist["id"]] = artist["name"]

print(f"\nCollected {len(artists)} artists.\nPrinting the first 25:")
for name in sorted(artists.values())[:25]:
    print(" ", name)

MusicBrainz lists 448 artists linked to the Faroe Islands

Collected 426 artists.
Printing the first 25:
  17 Sangarar
  200
  4
  48 Pages
  Albert Djurhuus
  Aldubáran
  All That Rain
  Allan Streymoy
  Andras & Jonfinn
  Anfinn & Co.
  Anna Faroe
  Anna Klett
  Anna Nielsen
  Anna í Kálvalíð
  Annika Hoydal
  Anny Leo Thorsen
  Aria
  Arnold Ludvig
  Arrestment
  Asyllex
  Atli Petersen
  Ave
  Axel Tórgarð
  Bardur Haberg
  Bartal Augustinussen
